# Explore MILK10k

The question for our first model is how well one clinical close-up can help classify a lesion. Run these cells in order to check the data, see which cases it represents, and compare the two image types. The final section turns the findings into decisions before training.

[ISIC dataset and attribution](https://api.isic-archive.com/doi/milk10k/) · [Diagnosis categories](https://challenge.isic-archive.com/landing/milk10k/#task)

## 1. Load the training files

The metadata and supplement have one row per image. Ground truth has one row per lesion. The `MILK10K_DATA_DIR` override is useful when Git worktrees share one download.

In [ ]:
import hashlib
import os
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image

default_data_dir = Path("data") if Path("pyproject.toml").is_file() else Path("../data")
data_dir = Path(os.environ.get("MILK10K_DATA_DIR", str(default_data_dir))).expanduser().resolve()
image_dir = data_dir / "MILK10k_Training_Input"
csv_paths = [data_dir / f"MILK10k_Training_{name}.csv" for name in ("Metadata", "GroundTruth", "Supplement")]

missing = [path.name for path in [image_dir, *csv_paths] if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing {', '.join(missing)} in {data_dir}. Add the MILK10k files or set MILK10K_DATA_DIR.")

metadata, ground_truth, supplement = [pd.read_csv(path) for path in csv_paths]
print(f"{len(metadata):,} image metadata rows, {len(ground_truth):,} lesion labels, {len(supplement):,} supplement rows")

## 2. Check the joins and count each lesion once

Each lesion should have one clinical close-up and one dermoscopic image. A clinical close-up is closer to our intended phone input. The dermoscopic view uses a specialized instrument.

The diagnosis columns use one-hot encoding, with a single 1 marking the category. Keep the original MILK10k code and look up a readable name for display. The supplement's confirmation method is used for auditing labels.

In [ ]:
assert metadata["isic_id"].notna().all() and metadata["isic_id"].is_unique, "Image metadata needs a unique isic_id per row"
assert supplement["isic_id"].is_unique, "The supplement should have one row per image"
assert ground_truth["lesion_id"].is_unique, "Ground truth should have one row per lesion"
assert metadata["lesion_id"].notna().all(), "Every image needs a lesion_id"
assert set(metadata["isic_id"]) == set(supplement["isic_id"]), "Metadata and supplement image IDs differ"
assert set(metadata["lesion_id"]) == set(ground_truth["lesion_id"]), "Metadata and ground-truth lesion IDs differ"

label_columns = ground_truth.drop(columns="lesion_id")
assert label_columns.isin([0, 1]).all().all() and label_columns.sum(axis=1).eq(1).all(), "Expected one active label per lesion"

diagnosis_names = {
    "AKIEC": "Actinic keratosis or intraepidermal carcinoma",
    "BCC": "Basal cell carcinoma",
    "BEN_OTH": "Other benign growths",
    "BKL": "Benign keratinocytic lesion",
    "DF": "Dermatofibroma",
    "INF": "Inflammatory or infectious conditions",
    "MAL_OTH": "Other malignant growths",
    "MEL": "Melanoma",
    "NV": "Mole (melanocytic nevus)",
    "SCCKA": "Squamous cell carcinoma or keratoacanthoma",
    "VASC": "Blood vessel lesions or bleeding",
}
assert set(label_columns.columns) == diagnosis_names.keys(), "Diagnosis names must match the MILK10k codes"
labels = ground_truth[["lesion_id"]].assign(diagnosis_code=label_columns.idxmax(axis=1))
labels["diagnosis"] = labels["diagnosis_code"].map(diagnosis_names)

image_types = {"clinical: close-up": "clinical_path", "dermoscopic": "dermoscopic_path"}
assert metadata["image_type"].isin(image_types).all(), "Every image needs a known image type"
pair_counts = pd.crosstab(metadata["lesion_id"], metadata["image_type"])
assert set(pair_counts.columns) == set(image_types) and pair_counts.eq(1).all().all(), "Expected one image of each type per lesion"

images = metadata.merge(supplement[["isic_id", "diagnosis_confirm_type", "diagnosis_full"]], on="isic_id", validate="one_to_one")
images["path"] = [
    image_dir / lesion_id / f"{isic_id}.jpg" for lesion_id, isic_id in zip(images["lesion_id"], images["isic_id"], strict=True)
]
disk_paths = set(image_dir.glob("*/*.jpg"))
expected_paths = set(images["path"])
assert expected_paths == disk_paths, (
    f"Image files differ from metadata: {len(expected_paths - disk_paths)} missing, {len(disk_paths - expected_paths)} extra"
)

lesion_fields = ["age_approx", "sex", "skin_tone_class", "site", "diagnosis_confirm_type", "diagnosis_full"]
field_counts = images.groupby("lesion_id")[lesion_fields].nunique(dropna=False)
assert field_counts.eq(1).to_numpy().all(), "Some paired images disagree on lesion metadata"

paths = images.pivot(index="lesion_id", columns="image_type", values="path").rename(columns=image_types)
lesions = (
    labels.set_index("lesion_id")
    .join(paths, validate="one_to_one")
    .join(images.drop_duplicates("lesion_id").set_index("lesion_id")[lesion_fields], validate="one_to_one")
)
print(f"{len(lesions):,} lesions with both images and a diagnosis")

## 3. See which diagnoses and groups are represented

Look at the smallest classes first. A class with very few examples can have a high or low score because of just one prediction. Overall accuracy can also look good when a model mostly predicts the largest class.

Check absolute counts for each diagnosis and skin tone combination. Keep missing values visible. Compare how diagnoses were confirmed across classes, since the confidence we put in the labels depends in part on their source.

In [ ]:
diagnosis_counts = lesions["diagnosis"].value_counts()
ax = diagnosis_counts.sort_values().plot.barh(figsize=(10, 4.5), title="Lesions per diagnosis", xlabel="Lesions", ylabel="")
ax.bar_label(ax.containers[0], padding=4)
ax.margins(x=0.14)
plt.tight_layout()
plt.show()

majority_share = diagnosis_counts.max() / len(lesions)
print(f"Always predicting {diagnosis_counts.idxmax()} would score {majority_share:.1%} accuracy on this dataset.")

display(lesions[lesion_fields].isna().sum().rename("missing lesions").to_frame())

for column in ["skin_tone_class", "site", "age_approx", "sex"]:
    display(lesions[column].value_counts(dropna=False).rename("lesions").to_frame())

for column in ["skin_tone_class", "diagnosis_confirm_type"]:
    groups = lesions[column].astype("string").fillna("Missing")
    display(pd.crosstab(lesions["diagnosis"], groups))

### How does the diagnosis mix compare with teledermatology?

MILK10k was assembled to study lesions that could be malignant. Histopathology confirmed 95.7% of its cases, and the authors note that ordinary benign lesions are underrepresented. [MILK10k study](https://doi.org/10.1016/j.jid.2025.06.1594)

The São Paulo teledermatology study used dermatologists' assessments of photos taken by health technicians from 2017 to 2018. The chart uses the study's reported total of 17,233 lesions across ten diagnosis groups. The individual rows in the source table sum to 17,223. Each bar uses its full cohort denominator. [Giavina-Bianchi et al. (2020), Table 1](https://www.frontiersin.org/journals/medicine/articles/10.3389/fmed.2020.598903/full)

The study's nevus category is the closest match for melanocytic nevus. For seborrheic keratosis and actinic keratosis, use the specific diagnoses in the MILK10k supplement. Those are narrower than the BKL and AKIEC groups above. The other broad groups lack a clear match in the study table.

A population screening study has a different denominator. The German SCREEN study found 1,961 confirmed BCCs across 360,288 people screened, or about 5.4 BCCs per 1,000 people screened. The bars below measure the share of lesions within each study. To estimate the mix of app submissions, we would need a labeled sample of those submissions. [German SCREEN study](https://jamanetwork.com/journals/jamadermatology/fullarticle/1158556)

In [ ]:
# Counts and denominator from Giavina-Bianchi et al., Table 1.
reference_total = 17_233
reference_counts = pd.Series(
    {
        diagnosis_names["BCC"]: 418,
        diagnosis_names["NV"]: 4_351,
        diagnosis_names["MEL"]: 163,
        "Seborrheic keratosis": 3_035,
        "Actinic keratosis": 838,
    }
)

specific_counts = lesions["diagnosis_full"].value_counts()
milk_counts = pd.concat(
    [
        diagnosis_counts.loc[[diagnosis_names[code] for code in ("BCC", "NV", "MEL")]],
        specific_counts.loc[["Seborrheic keratosis", "Solar or actinic keratosis"]].rename(
            index={"Solar or actinic keratosis": "Actinic keratosis"}
        ),
    ]
)
shares = pd.DataFrame(
    {
        f"MILK10k (n={len(lesions):,})": milk_counts / len(lesions) * 100,
        f"São Paulo, 10 diagnosis groups (n={reference_total:,})": reference_counts / reference_total * 100,
    }
)

ax = shares.plot.barh(figsize=(9, 4), title="Diagnosis shares in two lesion cohorts", xlabel="Share of lesions (%)")
ax.invert_yaxis()
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.1f%%", padding=4)
ax.margins(x=0.18)
ax.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

## 4. Check image quality and exact duplicates

The next cell decodes each JPEG and compares dimensions between modalities. Very different sizes and aspect ratios can affect resizing or cropping. It also hashes the image files to find byte-identical copies. If the same file occurs under different lesion IDs, review those cases before splitting the dataset.

Re-encoded or cropped copies can have different hashes. Keep both photos of each lesion together in every split. The supplied CSVs lack patient identifiers, so we cannot verify patient-level separation with these files alone.

In [ ]:
image_stats = []
unreadable = []
for isic_id, path in zip(images["isic_id"], images["path"], strict=True):
    try:
        content = path.read_bytes()
        with Image.open(BytesIO(content)) as photo:
            photo.load()
            width, height = photo.size
        image_stats.append((isic_id, width, height, width / height, hashlib.sha256(content).hexdigest()))
    except OSError as error:
        unreadable.append((isic_id, str(error)))

print(f"{len(unreadable):,} unreadable images out of {len(images):,}")
if unreadable:
    display(pd.DataFrame(unreadable, columns=["isic_id", "error"]).head(10))

stats = pd.DataFrame(image_stats, columns=["isic_id", "width", "height", "aspect_ratio", "sha256"])
stats = stats.merge(images[["isic_id", "lesion_id", "image_type"]], on="isic_id", validate="one_to_one")
display(stats.groupby("image_type")[["width", "height", "aspect_ratio"]].agg(["min", "median", "max"]).round(2))

lesions_per_hash = stats.groupby("sha256")["lesion_id"].transform("nunique")
duplicates = stats.loc[lesions_per_hash.gt(1)].sort_values(["sha256", "lesion_id"])
print(f"{duplicates['sha256'].nunique():,} exact duplicate groups across different lesion IDs")
if not duplicates.empty:
    display(duplicates[["lesion_id", "isic_id", "image_type", "sha256"]].head(12))

## 5. Look at paired examples

Try changing `gallery_codes` using the keys in `diagnosis_names`, or try a different random seed. Look for blur, lighting, framing, rulers, pen marks, and differences in background. Ask whether those cues could correlate with diagnosis in this dataset or disappear when someone takes a photo at home.

Clear all notebook outputs before committing so the dataset images stay local.

In [ ]:
gallery_codes = ["BCC", "MEL", "MAL_OTH"]
unreadable_lesions = images.loc[~images["isic_id"].isin(stats["isic_id"]), "lesion_id"]
candidates = lesions.loc[lesions["diagnosis_code"].isin(gallery_codes) & ~lesions.index.isin(unreadable_lesions)]
assert not candidates.empty, "Choose a diagnosis with at least one readable pair"
examples = candidates.groupby("diagnosis_code").sample(1, random_state=42)

fig, axes = plt.subplots(len(examples), 2, figsize=(10, 3 * len(examples)), squeeze=False)
for row_axes, example in zip(axes, examples.itertuples(), strict=True):
    pairs = [(example.clinical_path, "Clinical close-up"), (example.dermoscopic_path, "Dermoscopy")]
    for ax, (path, modality) in zip(row_axes, pairs, strict=True):
        with Image.open(path) as photo:
            photo.thumbnail((480, 480))
            ax.imshow(photo.convert("RGB"))
        ax.set_title(f"{example.diagnosis}\n{modality}")
        ax.axis("off")

fig.tight_layout()
plt.show()

## What to decide before training

- Which rare diagnoses and skin tone groups have enough cases to report meaningful results? Plan per-class scores and include the underlying counts.
- Can we keep known related examples in one split, and what overlap could still go undetected?
- Which fields could we actually get when a person takes a photo? Start with the clinical image alone. Use identifiers to join data and the diagnostic supplement to understand the labels.
- What might change from clinical close-ups to self-taken photos? Use the gallery to list likely differences and plan an independent evaluation on self-taken images.